In [ ]:
import os
import pandas as pd
from glob import glob
from nilearn.glm.first_level import first_level_from_bids
from nilearn.interfaces.fmriprep import load_confounds

## nilearn modeling: first level


Based on [nilearn examples](https://nilearn.github.io/auto_examples/04_glm_first_level/plot_bids_features.html#sphx-glr-auto-examples-04-glm-first-level-plot-bids-features-py)

In [ ]:
# Rename events based on desired analysis
def update_events(models_events, event_type='sound'):
    # SNR-level events (badaga task's speech-in-noise conditions)
    if event_type == 'snr':
        for sx, sub_events in enumerate(models_events):
            for mx, run_events in enumerate(sub_events):
                run_events['trial_type'] = run_events['noise_level']

        # create stimulus list from updated events.tsv file
        stim_list = sorted([str(s) for s in run_events['trial_type'].unique() if str(s) not in ['nan', 'None']])

    # all sound events, collapsed across trial-level suffixes
    elif event_type == 'sound':
        for sx, sub_events in enumerate(models_events):
            for mx, run_events in enumerate(sub_events):
                orig_stim_list = sorted([str(s) for s in run_events['trial_type'].unique() if str(s) not in ['nan', 'None']])
                #print('original stim list: ', orig_stim_list)

                run_events['trial_type'] = run_events.trial_type.str.split('_', expand=True)[0]

        # create stimulus list from updated events.tsv file
        stim_list = sorted([str(s) for s in run_events['trial_type'].unique() if str(s) not in ['nan', 'None']])

    else:
        # previously 'stimulus'/'trial' branches existed here but had no matching contrast_list
        # in nilearn_glm_across_runs() (NameError if used) and were unused/vestigial; removed.
        raise ValueError(f"No trial_type construction defined for event_type={event_type!r} "
                          "(only 'snr' and 'sound' are supported)")

    #print('stim list: ', stim_list)
    return stim_list, models_events

In [ ]:
# Across-runs GLM
def nilearn_glm_across_runs(stim_list, task_label,
                            models, models_run_imgs,
                            models_events,
                            models_confounds,
                            event_type,
                            out_dir):
    from nilearn.interfaces.bids import save_glm_to_bids
    from nilearn.interfaces.fmriprep import load_confounds_strategy

    midx = 0 # only 1 subject per analysis

    if event_type == 'sound':
        contrast_list = ['sound', 'response']
    elif event_type == 'snr':
        contrast_list = ['Q', '8', '0', 'n2', 'n6', 'Q - 0', 'Q - n6']
    else:
        raise ValueError(f"No contrast_list defined for event_type={event_type!r} "
                          "(only 'snr' and 'sound' are supported)")

    model = models[midx]
    imgs = models_run_imgs[midx]
    events = models_events[midx]

    # Select confounds + a motion-based censoring mask ONCE per subject. Previously this (and the
    # GLM fit below) ran once per contrast inside the loop, refitting the same model from scratch
    # up to 7x per subject for no reason. 'scrubbing' adds FD/DVARS-based volume censoring on top
    # of compcor + motion regressors; sample_mask is now actually passed into model.fit() below
    # instead of being computed and silently discarded.
    # FD/DVARS thresholds here are nilearn's 'scrubbing' preset defaults (fd_threshold=0.5mm,
    # std_dvars_threshold=1.5) -- confirm with the PI whether these are appropriate for this
    # pediatric/stuttering population, where task-related orofacial/vocal motion during speech
    # trials may differ from adult norms.
    print('selecting confounds and motion-scrubbing mask')
    confounds_ltd, sample_mask = load_confounds_strategy(img_files=imgs,
                                                         denoise_strategy='scrubbing')

    # Log per-subject mean framewise displacement, using the raw per-run fMRIPrep confounds
    # tables first_level_from_bids already loaded into models_confounds (previously passed into
    # this function but never used). 'framewise_displacement' is a raw fMRIPrep confound column,
    # not part of confounds_ltd's denoise-strategy-filtered regressor set. This manifest feeds
    # the group-level motion covariate.
    raw_confounds = models_confounds[midx]
    fd_by_run = []
    for rc in raw_confounds:
        rc_df = rc if isinstance(rc, pd.DataFrame) else pd.read_csv(rc, sep='\t')
        fd_by_run.append(rc_df['framewise_displacement'].mean())

    motion_qc_row = pd.DataFrame([{
        'subject_id': f'sub-{model.subject_label}',
        'mean_fd': pd.Series(fd_by_run).mean(),
        **{f'mean_fd_run-{rx + 1}': fd for rx, fd in enumerate(fd_by_run)},
    }])
    # One file per subject, not a shared appended motion_qc.csv: univariate_first-level.py
    # runs this same function once per subject via parallel SLURM jobs, which would otherwise
    # all be appending to the same file with no locking in place. The notebook's own per-subject
    # loop is sequential, so this was never actually racy here, but both entry points write the
    # same layout the group-level notebook expects.
    subject_out_dir = os.path.join(out_dir, f'sub-{model.subject_label}')
    os.makedirs(subject_out_dir, exist_ok=True)
    motion_qc_fpath = os.path.join(subject_out_dir, f'sub-{model.subject_label}_motion_qc.csv')
    motion_qc_row.to_csv(motion_qc_fpath, index=False)

    # fit the GLM once
    print('fitting GLM')
    # NOTE: the keyword for the censoring mask changed between nilearn versions
    # ('sample_mask' singular pre-0.10, 'sample_masks' plural for multi-run fits in newer
    # releases). Confirm this matches the nilearn version installed on the cluster.
    model.fit(imgs, events, confounds_ltd, sample_masks=sample_mask)
    print(model)

    for contrast_label in contrast_list:
        print('Running for contrast', contrast_label)

        # compute the contrast of interest
        print('computing contrast of interest')
        summary_statistics = model.compute_contrast(contrast_label, output_type='all')

        # save model outputs
        out_prefix = f"sub-{model.subject_label}_task-{task_label}_fwhm-{model.smoothing_fwhm}"
        save_glm_to_bids(model,
                         contrast_label,
                         out_dir=out_dir,
                         prefix=out_prefix,
                        )
        print(f'Saved model outputs to {out_dir}')

    return summary_statistics

## Run pipelines

In [ ]:
task_label  = 'badaga'
space_label = 'MNI152NLin2009cAsym'
event_type  = 'snr' # 'snr', 'sound'

t_acq = 2
t_r = t_acq # same as t_acq since no silent gap in this acquisition

# NOTE: slice_time_ref is intentionally NOT computed here. It used to be hand-derived as
# 0.5 * t_acq / t_r (a "middle slice" guess for a naive continuous acquisition), but that guess
# doesn't match this dataset's actual slice-timing protocol -- nilearn warned that the provided
# value (0.5) differed from what it read from the BIDS bold.json metadata (0.174). Passing
# slice_time_ref=None below lets first_level_from_bids infer the correct value directly from
# that metadata instead of silently overriding it with a wrong hand-computed one.

# define bids and fmriprep directories
project_dir = os.path.join('/bgfs/bchandrasekaran/krs228/data/', 
                           'SSP/')
bidsroot = os.path.join(project_dir, 
                        'data_bids')
fmriprep_dir = os.path.join(bidsroot, 
                            'derivatives', 
                            'fmriprep-23.2.1',
                            )
print('bidsroot: ', bidsroot)
print('fmriprep dir:', fmriprep_dir)

# create output directory
if event_type == 'snr':
    bidsderiv_dir = os.path.join(bidsroot, 
                                 'derivatives', 
                                 'nilearn', 
                                 'run-all_contrast-snr')
else:
    bidsderiv_dir = os.path.join(bidsroot, 
                                 'derivatives', 
                                 'nilearn', 
                                 'run-all')
os.makedirs(bidsderiv_dir, exist_ok=True)

### Univariate analysis

**Modeling choices (recorded for reproducibility):** `fwhm=6` (smoothing kernel), and the HRF model (`'glover'`, no temporal derivative) and drift model (`'cosine'`, 128s high-pass) are left at `first_level_from_bids`'s implicit defaults rather than set explicitly. These are standard choices for this design, but leaving them implicit ties the analysis to whatever `first_level_from_bids` defaults to in the `nilearn` version installed on the cluster at run time -- if reproducibility across `nilearn` versions matters, set them explicitly (`hrf_model=`, `drift_model=`) instead of relying on the default.

In [5]:
fwhm = 6
'''
subject_list = [os.path.basename(x).split('-')[1] 
                for x in sorted(os.listdir(bidsroot)) 
                if 'sub' in x
                ] # 
'''
subject_list = ['SSP096', 'SSP105', 'SSP106', 'SSP107', 'SSP108', 'SSP109', 
                'SSP110', 'SSP111', 'SSP112', 'SSP113', 'SSP114', 
                ]


In [ ]:
run_log = []

for sx, subject_id in enumerate(subject_list):
    print('Running subject', subject_id)
    try:
        models, models_run_imgs, \
                raw_models_events, \
                models_confounds = first_level_from_bids(bidsroot, 
                                                         task_label, 
                                                         space_label=space_label,
                                                         sub_labels=[subject_id],
                                                         smoothing_fwhm=fwhm,
                                                         derivatives_folder=fmriprep_dir,
                                                         slice_time_ref=None,  # infer from BIDS metadata, see note above
                                                         minimize_memory=False)


        stim_list, models_events = update_events(raw_models_events, 
                                                 event_type=event_type)

        # Across-run GLM
        summary_statistics = nilearn_glm_across_runs(stim_list, 
                                                     task_label, 
                                                     models, 
                                                     models_run_imgs, 
                                                     models_events, 
                                                     models_confounds,
                                                     event_type,
                                                     out_dir=bidsderiv_dir)
        run_log.append({'subject_id': subject_id, 'status': 'ok',
                        'error_type': None, 'error_msg': None})
    except Exception as e:
        # Previously: `except KeyError: print(...)` then a bare `except: continue`, which
        # silently swallowed every other failure (missing confound file, mismatched events,
        # GLM fit errors, ...) with no record of which subjects failed or why, and also masked
        # KeyboardInterrupt/SystemExit via the bare except. Now every failure is logged to a
        # manifest CSV instead.
        print(f'FAILED for {subject_id}: {type(e).__name__}: {e}')
        run_log.append({'subject_id': subject_id, 'status': 'failed',
                        'error_type': type(e).__name__, 'error_msg': str(e)})

run_log_df = pd.DataFrame(run_log)
run_log_fpath = os.path.join(bidsderiv_dir, 'first_level_run_manifest.csv')
run_log_df.to_csv(run_log_fpath, index=False)
n_ok = (run_log_df['status'] == 'ok').sum()
n_failed = (run_log_df['status'] == 'failed').sum()
print(f'\nFirst-level run summary: {n_ok} succeeded, {n_failed} failed. '
     f'See {run_log_fpath} for details.')

In [ ]:
print(stim_list)